# 01 - Data Exploration et Prétraitement

In [1]:
#Importation des bibliothèques 

import os
import re
import glob
import PyPDF2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

# Téléchargement des ressources NLTK 
nltk.download('punkt')
nltk.download('stopwords')

print(" Bibliothèques chargées")

 Bibliothèques chargées


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
INPUT_DIR = "../data/raw"          # dossier contenant les CSV sources (biblio.csv, formation.csv, etc.)
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Stop words français étendus
STOP_WORDS = set(stopwords.words('french')).union({
    'ce', 'cet', 'cette', 'ces', 'mon', 'ton', 'son', 'notre', 'votre', 'leur',
    'mais', 'ou', 'et', 'donc', 'or', 'ni', 'car', 'je', 'tu', 'il', 'elle',
    'nous', 'vous', 'ils', 'elles', 'me', 'te', 'se', 'lui', 'leur', 'y', 'en',
    'est', 'sont', 'suis', 'es', 'sommes', 'êtes', 'ai', 'as', 'a', 'avons', 'avez', 'ont',
    'comment', 'pourquoi', 'quand', 'où', 'quel', 'quelle', 'quels', 'quelles',
    'très', 'trop', 'peu', 'assez', 'presque', 'environ'
})

stemmer = SnowballStemmer('french')

print(f" Dossier d'entrée : {INPUT_DIR}")
print(f" Dossier de sortie : {OUTPUT_DIR}")

 Dossier d'entrée : ../data/raw
 Dossier de sortie : ../data/processed


In [ ]:
# Chargement et fusion des CSV

def load_csv_files(directory):
    """Charge tous les fichiers CSV d’un dossier et les fusionne"""
    all_files = glob.glob(os.path.join(directory, "*.csv"))
    if not all_files:
        raise FileNotFoundError(f"Aucun fichier CSV trouvé dans {directory}")
    
    dataframes = []
    for f in all_files:
        df = pd.read_csv(f, sep=';', encoding='utf-8')
        print(f" Chargé : {os.path.basename(f)} ({len(df)} lignes)")
        dataframes.append(df)
    
    return pd.concat(dataframes, ignore_index=True)

try:
    df_raw = load_csv_files(INPUT_DIR)
except pd.errors.ParserError as e:
    print("ParserError during initial load, retrying with flexible parsing:", e)
    all_files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))
    dfs = []
    for f in all_files:
        try:
            # try strict ';' with skip on bad lines
            df = pd.read_csv(f, sep=';', encoding='utf-8', engine='python', on_bad_lines='skip', low_memory=False)
            print(f" Chargé : {os.path.basename(f)} ({len(df)} lignes) [sep=';']")
        except Exception:
            try:
                # fallback: let pandas infer the separator
                df = pd.read_csv(f, sep=None, encoding='utf-8', engine='python', on_bad_lines='skip', low_memory=False)
                print(f" Chargé : {os.path.basename(f)} ({len(df)} lignes) [inferred sep]")
            except Exception as e2:
                print(f" Échec lecture {os.path.basename(f)} : {e2}")
                continue
        dfs.append(df)
    if not dfs:
        raise FileNotFoundError(f"Aucun fichier CSV valide trouvé dans {INPUT_DIR}")
    df_raw = pd.concat(dfs, ignore_index=True)
print(f"\n Total : {len(df_raw)} lignes")
display(df_raw.head())

ParserError: Error tokenizing data. C error: Expected 2 fields in line 6, saw 8
